In [ ]:
import os
import re
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

load_dotenv(override=True)

OPEN_ROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")
OPEN_ROUTER_BASE_URL = "https://openrouter.ai/api/v1"

openrouter = OpenAI(
    base_url=OPEN_ROUTER_BASE_URL,
    api_key=OPEN_ROUTER_API_KEY,
)

OPENROUTER_MODELS = [
    "openai/gpt-4o-mini",
    "google/gemini-2.0-flash-001",
    "anthropic/claude-3.5-haiku",
]

SEED = 42

In [ ]:
def build_text(row: Dict) -> str:
    title = (row.get("title") or "").strip()
    features = " ".join(row.get("features") or [])
    description = " ".join(row.get("description") or [])
    details = row.get("details")
    if isinstance(details, dict):
        details_text = " ".join(f"{k}: {v}" for k, v in details.items())
    else:
        details_text = str(details or "")
    text = f"{title}. {features} {description} {details_text}".strip()
    text = re.sub(r"\s+", " ", text)
    return text


def clean_row(row: Dict, min_chars: int = 40, max_chars: int = 1400) -> Optional[Dict]:
    try:
        price = float(row.get("price"))
    except (TypeError, ValueError):
        return None

    if not (1 <= price <= 1000):
        return None

    text = build_text(row)
    if len(text) < min_chars:
        return None

    return {
        "text": text[:max_chars],
        "price": price,
    }


def load_items(category: str = "raw_meta_Appliances", max_rows: int = 12000) -> List[Dict]:
    dataset = load_dataset(
        "McAuley-Lab/Amazon-Reviews-2023",
        category,
        split="full",
        trust_remote_code=True,
    )

    if max_rows and len(dataset) > max_rows:
        dataset = dataset.select(range(max_rows))

    rows = [clean_row(x) for x in dataset]
    return [r for r in rows if r is not None]

## 2) Train fast traditional ML model (Ridge + bag of words)

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def train_fast_model(items: List[Dict], test_size: float = 0.2, max_features: int = 2500):
    texts = [x["text"] for x in items]
    prices = np.array([x["price"] for x in items], dtype=float)

    x_train_text, x_test_text, y_train, y_test = train_test_split(
        texts,
        prices,
        test_size=test_size,
        random_state=SEED,
    )

    vectorizer = CountVectorizer(max_features=max_features, stop_words="english")
    x_train = vectorizer.fit_transform(x_train_text)
    x_test = vectorizer.transform(x_test_text)

    model = Ridge(alpha=1.0, random_state=SEED)
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)

    baseline = float(np.mean(y_train))
    baseline_pred = np.full(shape=len(y_test), fill_value=baseline)

    results = {
        "baseline_mae": float(mean_absolute_error(y_test, baseline_pred)),
        "baseline_rmse": rmse(y_test, baseline_pred),
        "ml_mae": float(mean_absolute_error(y_test, y_pred)),
        "ml_rmse": rmse(y_test, y_pred),
        "train_size": len(y_train),
        "test_size": len(y_test),
        "baseline_value": baseline,
    }

    test_df = pd.DataFrame({
        "text": x_test_text,
        "actual": y_test,
        "ml_pred": y_pred,
    })
    return model, vectorizer, results, test_df

In [ ]:
CATEGORY = "raw_meta_Appliances"
MAX_ROWS = 12000

items = load_items(category=CATEGORY, max_rows=MAX_ROWS)
print(f"Clean items: {len(items):,}")

model, vectorizer, metrics, test_df = train_fast_model(items, test_size=0.2, max_features=2500)

print("\nTraditional ML results")
print(f"Train size: {metrics['train_size']:,}")
print(f"Test size : {metrics['test_size']:,}")
print(f"Baseline MAE : ${metrics['baseline_mae']:.2f}")
print(f"Model MAE    : ${metrics['ml_mae']:.2f}")
print(f"Baseline RMSE: ${metrics['baseline_rmse']:.2f}")
print(f"Model RMSE   : ${metrics['ml_rmse']:.2f}")

display(test_df.head(10))

## 3) Optional OpenRouter LLM predictor (for comparison)

In [ ]:
PRICE_SYSTEM_PROMPT = (
    "You are a pricing assistant. Given a product description, estimate the price in USD. "
    "Return only a number, no explanation and no currency symbol."
)


def parse_price(text: str, fallback: float) -> float:
    if not text:
        return fallback
    match = re.search(r"\d+(?:\.\d+)?", text.replace(",", ""))
    if not match:
        return fallback
    try:
        return float(match.group(0))
    except ValueError:
        return fallback


def predict_with_openrouter(description: str, model_name: str = OPENROUTER_MODELS[0], temperature: float = 0.2) -> float:
    if not OPEN_ROUTER_API_KEY:
        raise ValueError("OPEN_ROUTER_API_KEY is not set.")

    response = openrouter.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": PRICE_SYSTEM_PROMPT},
            {"role": "user", "content": f"Product description:\n{description}\n\nPrice:"},
        ],
        temperature=temperature,
        max_tokens=20,
    )

    content = (response.choices[0].message.content or "").strip()
    return max(0.0, parse_price(content, fallback=metrics["baseline_value"]))


def evaluate_llm_sample(sample_size: int = 20, model_name: str = OPENROUTER_MODELS[0]):
    if sample_size <= 0:
        return None
    subset = test_df.sample(n=min(sample_size, len(test_df)), random_state=SEED).copy()
    preds = []
    for text in subset["text"]:
        preds.append(predict_with_openrouter(text, model_name=model_name, temperature=0.2))
    subset["llm_pred"] = preds
    llm_mae = mean_absolute_error(subset["actual"], subset["llm_pred"])
    print(f"OpenRouter sample MAE on {len(subset)} rows: ${llm_mae:.2f}")
    return subset

# Example (optional, uses API credits):
# llm_eval_df = evaluate_llm_sample(sample_size=15, model_name=OPENROUTER_MODELS[0])
# display(llm_eval_df.head())

## 4) Gradio app (Plan 2)

In [ ]:
def predict_ml_price(description: str) -> float:
    x = vectorizer.transform([description])
    pred = float(model.predict(x)[0])
    return max(0.0, pred)


def app_predict(
    description: str,
    call_llm: bool,
    llm_model: str,
    temperature: float,
):
    if not description or not description.strip():
        return "Enter a product description.", "", "", ""

    description = description.strip()
    baseline_pred = float(metrics["baseline_value"])
    ml_pred = predict_ml_price(description)

    llm_result = "Skipped"
    llm_error = ""
    if call_llm:
        try:
            llm_pred = predict_with_openrouter(description, model_name=llm_model, temperature=temperature)
            llm_result = f"${llm_pred:.2f}"
        except Exception as e:
            llm_result = "Failed"
            llm_error = f"LLM error: {e}"

    summary = (
        f"Baseline (mean): ${baseline_pred:.2f}\n"
        f"Traditional ML (Ridge): ${ml_pred:.2f}\n"
        f"OpenRouter LLM: {llm_result}"
    )

    return summary, f"${baseline_pred:.2f}", f"${ml_pred:.2f}", llm_error


with gr.Blocks(title="Week 6 - Price Predictor") as demo:
    gr.Markdown("## Week 6 Price Predictor (Plan 1 + 2)")
    gr.Markdown("Fast CPU-friendly ML model + optional OpenRouter comparison")

    description_in = gr.Textbox(
        label="Product Description",
        lines=8,
        placeholder="Paste product title/features/description here...",
    )

    with gr.Row():
        call_llm_in = gr.Checkbox(label="Also query OpenRouter LLM", value=False)
        llm_model_in = gr.Dropdown(choices=OPENROUTER_MODELS, value=OPENROUTER_MODELS[0], label="LLM model")
        temperature_in = gr.Slider(0.0, 1.0, value=0.2, step=0.1, label="Temperature")

    predict_btn = gr.Button("Predict")

    summary_out = gr.Textbox(label="Prediction Summary", lines=5)
    with gr.Row():
        baseline_out = gr.Textbox(label="Baseline")
        ml_out = gr.Textbox(label="Traditional ML")
    error_out = gr.Textbox(label="Errors / Notes", lines=2)

    predict_btn.click(
        fn=app_predict,
        inputs=[description_in, call_llm_in, llm_model_in, temperature_in],
        outputs=[summary_out, baseline_out, ml_out, error_out],
    )

# If localhost access is restricted in your environment, use demo.launch(share=True)
demo.launch()